In [ ]:
# Locate the project root (where config.yaml lives) and set as cwd
import os
from pathlib import Path
for _parent in [Path.cwd(), Path.cwd().parent]:
    if (_parent / 'config.yaml').exists():
        os.chdir(_parent)
        break
print('Working directory:', os.getcwd())


# Step 01: FastQC quality control


Runs FastQC on every FASTQ file, then aggregates all reports into a single MultiQC summary.


**Before running:** open `Data/samplesheet.csv` and check that the `group` column is correct (e.g. Control, Empty, siKRAS, siScr). This column is not used by FastQC, but it must be correct before running differential expression (Step 05).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import yaml
from utils.helpers import launch_background

cfg = yaml.safe_load(open('config.yaml'))
launch_background(
    [sys.executable, 'scripts/fastqc.py',
     '--samplesheet', 'Data/samplesheet.csv',
     '--output-dir', 'Data/fastqc',
     '--fastqc-bin', cfg['tools']['fastqc_bin']],
    'Data/logs/01_fastqc.log'
)


## MultiQC summary


After FastQC finishes (monitor with `!tail -f Data/logs/01_fastqc.log`), run the cell below to aggregate all 24 reports into one summary HTML.


In [ ]:
import subprocess
subprocess.run(['multiqc', 'Data/fastqc/', '-o', 'Data/multiqc',
                '-n', 'multiqc_report', '--force'])
print('MultiQC report -> Data/multiqc/multiqc_report.html')
